In [31]:
# Cell 1: ตรวจสอบ dataset จริง (ต้องมีภาพใน ./dataset/open และ ./dataset/closed)
import os, glob
import cv2
import numpy as np

EXTS = ('.png', '.jpg', '.jpeg', '.bmp')

def list_images(folder):
    return [p for p in glob.glob(os.path.join(folder, '*')) if p.lower().endswith(EXTS)]

for d in ('./dataset/open', './dataset/closed'):
    if not os.path.isdir(d):
        raise FileNotFoundError(f"ไม่พบโฟลเดอร์ {d} -> กรุณาวางภาพจริงไว้ในโฟลเดอร์นี้")

n_open = len(list_images('./dataset/open'))
n_closed = len(list_images('./dataset/closed'))
print(f"พบภาพ open={n_open}, closed={n_closed}")

if n_open == 0 or n_closed == 0:
    raise RuntimeError("dataset ว่าง หรือมีคลาสใดคลาสหนึ่งไม่มีภาพ -> ใส่ภาพจริงให้ครบทั้ง open และ closed ก่อนเทรน")


พบภาพ open=2000, closed=2000


In [32]:
# Cell 2: โหลดภาพ + preprocess (ต้องเหมือนใน main.py: grayscale -> equalizeHist -> resize 64x64 -> /255)
IMG_SIZE = 64

def preprocess(gray):
    gray = cv2.equalizeHist(gray)
    gray = cv2.resize(gray, (IMG_SIZE, IMG_SIZE)).astype('float32') / 255.0
    return gray[..., None]

X, y = [], []
for label, name in [(0, 'closed'), (1, 'open')]:      # closed=0, open=1
    for p in list_images(f'./dataset/{name}'):
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        X.append(preprocess(img)); y.append(label)

X = np.array(X); y = np.array(y, dtype='float32')
print(X.shape, "closed:", int((y == 0).sum()), "open:", int((y == 1).sum()))

(4000, 64, 64, 1) closed: 2000 open: 2000


In [33]:
# Cell 3: แบ่ง train/validation
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_val.shape)

(3200, 64, 64, 1) (800, 64, 64, 1)


In [34]:
# Cell 4: สร้างและเทรน CNN
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),

    layers.Conv2D(32, 3, activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation='relu'), layers.BatchNormalization(), layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dense(1, activation='sigmoid'),   # ค่าเข้าใกล้ 1 = ตาเปิด, ใกล้ 0 = ตาปิด
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30, batch_size=32,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ])

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_6 (RandomFlip)      │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_6               │ (None, 64, 64, 1)      │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_6 (RandomZoom)      │ (None, 64, 64, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 62, 62, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 62, 62, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_18 (MaxPooling2D) │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 29, 29, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_19 (MaxPooling2D) │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_20 (Conv2D)              │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_20          │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 64)             │       294,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 388,609 (1.48 MB)

 Trainable params: 388,161 (1.48 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 11s 91ms/step - accuracy: 0.9750 - loss: 0.0673 - val_accuracy: 0.5838 - val_loss: 0.5549 - learning_rate: 0.0010
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 88ms/step - accuracy: 0.9931 - loss: 0.0183 - val_accuracy: 0.8637 - val_loss: 0.3001 - learning_rate: 0.0010
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 86ms/step - accuracy: 0.9944 - loss: 0.0176 - val_accuracy: 0.9762 - val_loss: 0.0485 - learning_rate: 0.0010
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.9969 - loss: 0.0129 - val_accuracy: 0.9787 - val_loss: 0.0547 - learning_rate: 0.0010
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.9994 - loss: 0.0023 - val_accuracy: 0.9887 - val_loss: 0.0346 - learning_rate: 0.0010
Epoch 6/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.9991 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 7.9942e-05 - learning_rate: 0.0010
Epoch 7/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 9s 85ms/step - accuracy: 0.9987 - los

In [36]:
# Cell 5: ประเมินผล + บันทึกโมเดล (วางไฟล์ไว้โฟลเดอร์เดียวกับ main.py)
from sklearn.metrics import classification_report, confusion_matrix
pred = (model.predict(X_val, verbose=0)[:, 0] >= 0.5).astype(int)
print(confusion_matrix(y_val, pred))
print(classification_report(y_val, pred, target_names=['closed', 'open']))

model.save('eye_state_cnn.h5')
print("บันทึก eye_state_cnn.h5 เรียบร้อย")

[[400   0]
 [  0 400]]
              precision    recall  f1-score   support

      closed       1.00      1.00      1.00       400
        open       1.00      1.00      1.00       400

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800

บันทึก eye_state_cnn.h5 เรียบร้อย
